## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy

from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingClassifier

RANDOM_STATE = 0

---
## 1.
> Suppose that $Y$ is in the natural numbers. You decide to model $Y$ given $X$ with boosting. As your base model, you decide to use regression trees $\{h_j\}$ that take in covariates $X$ and output real valued entries. Then,

### (a)
> You decide to make your model $P(Y \mid X) \sim \mathrm{Poi}(\exp(f(X)))$, where $f(X) = \sum_j \lambda_j h_j(X)$, with each $h_j(X)$ a (trained) regression tree.

#### i.
> Why is Poisson a reasonable choice here?

Since $Y$ is a natural number, i.e. count data, and the Poisson is the standard model for counts, it's a reasonable choice. It also works well with boosting because its loss is convex and the gradient is simple, see (b) and (c).

#### ii.
> Why do we use exp() as our link function?

The Poisson distribution depends on a parameter $\mu > 0$ and since $f(X) \in \mathbb{R}$ is unrestricted, we ensure that our estimate of $\mu$ is greater than zero by using $\exp()$ as our link function.

### (b)
> Write down the loss function used for learning in terms of $f(x)$ and $Y$.

From
$$
p\big(Y \mid f(x)\big)
= \frac{ (e^{f(x)})^{Y}\, e^{-e^{f(x)}} }{Y!}
= \frac{e^{Y f(x)}\; e^{-e^{f(x)}}}{Y!},
$$
we get the per-example NLL
$$
\begin{aligned}
L\big(f(x), Y\big)
&= -\log p\big(Y \mid f(x)\big) \\[4pt]
&= e^{f(x)} - Y f(x) + \log(Y!).
\end{aligned}
$$

### (c)
> Give an expression for $\frac{dL}{df(x)}$.

$$
\frac{dL}{df(x)} = e^{f(x)} - Y
$$

### (d)
> Describe how you would choose each $h_j$ in (first order) boosting.

At step $j$, we compute the pseudo outcomes
$$
\gamma _{ij} = \left. - \frac{dL(f(x_i), y_i)}{df(x_i)} \right|_{f = f_{j-1}} = y_i - e^{f_{j-1}(x_i)}
$$
and then fit a regression tree that predicts $\gamma _{ij}$ given $x_i$,
$$
h_j = \arg\min _h \sum _{i=1}^n \left(\gamma _{ij} - h(x_i) \right)^2.
$$
Next, for step size $\lambda _j$ (fixed or chosen by line search), we update
$$
f_j = f_{j-1} + \lambda _j h_j.
$$
To compute the first residuals, we initialize $f_0$ as some constant. Under the Poisson loss $f_0 = \log (\bar{y})$ is a smart initialization, since this is the best possible model that ignores $X$.

---
## 2.
> Often, machine learning models involve so-called *hyperparameters*. These are quantities that are set by the analyst, rather than being learned by empirical risk minimization. For example, the regularization strength is a hyperparameter. One option for selecting hyperparameters is to try fitting a model with several different choices, and then using performance on a test set to select the best one.

### (a)
> Find (and prove) a non-trivial generalization bound for this procedure. That is, find a probabilistic bound on the gap between the risk estimated on the test set, and the true risk of the model. Give this bound in terms of (some subset of) the number of training examples $n$, the number of examples in the test set $l$, and the number of hyperparameters $k$.

**Setup.** Let $D^{\text{train}}$ (with $n$ observations) and $D^{\text{test}} = \{Z_1, \dots, Z_l\}$ be independent, where $Z_1, \dots, Z_l \overset{\text{iid}}{\sim} \mathbb{P}$. Let $\mathcal{K}$ be a set of $k = |\mathcal{K}|$ hyperparameter specifications, and for each $\kappa \in \mathcal{K}$ let $f_\kappa$ be the model fit on $D^{\text{train}}$ with hyperparameters $\kappa$. Let $L(z, f) \in [0, b]$ be a bounded loss, and define

$$
\mathcal{L}(f) = \mathbb{E}_{Z \sim \mathbb{P}}\big[L(Z, f)\big],
\qquad
\hat{\mathcal{L}}(f) = \frac{1}{l} \sum_{i=1}^{l} L(Z_i, f),
$$

the risk and the empirical risk on the test set. The selected model is $\hat f = f_{\hat\kappa}$ with $\hat\kappa \in \arg\min_{\kappa \in \mathcal{K}} \hat{\mathcal{L}}(f_\kappa)$.

**Claim.** For every $t > 0$,

$$
\mathbb{P}\Big( \big|\mathcal{L}(\hat f) - \hat{\mathcal{L}}(\hat f)\big| > t \Big) \le 2k \exp\!\left( -\frac{2 l t^2}{b^2} \right).
$$

Equivalently, for any $\delta \in (0, 1)$, with probability at least $1 - \delta$,

$$
\big|\mathcal{L}(\hat f) - \hat{\mathcal{L}}(\hat f)\big| \le b \sqrt{\frac{\log(2k/\delta)}{2l}}.
$$

**Proof.** Condition on $D^{\text{train}}$. Then each $f_\kappa$ is a fixed function. Since $D^{\text{test}}$ is independent of $D^{\text{train}}$, the $Z_i$ are still iid $\sim \mathbb{P}$ conditional on $D^{\text{train}}$.

Note that we cannot apply Hoeffding's inequality to $\hat f$ directly: $\hat f$ is selected using $D^{\text{test}}$, so the $L(Z_i, \hat f)$ are not independent and need not have mean $\mathcal{L}(\hat f)$. Instead, we use that $\hat f \in \{f_\kappa : \kappa \in \mathcal{K}\}$, so if the gap is large for $\hat f$, it is large for at least one $f_\kappa$:

$$
\Big\{ \big|\mathcal{L}(\hat f) - \hat{\mathcal{L}}(\hat f)\big| > t \Big\}
\subseteq
\bigcup_{\kappa \in \mathcal{K}} \Big\{ \big|\mathcal{L}(f_\kappa) - \hat{\mathcal{L}}(f_\kappa)\big| > t \Big\}.
$$

Hence

$$
\begin{aligned}
\mathbb{P}\Big( \big|\mathcal{L}(\hat f) - \hat{\mathcal{L}}(\hat f)\big| > t \,\Big|\, D^{\text{train}} \Big)
&\le \mathbb{P}\Big( \exists\, \kappa \in \mathcal{K} : \big|\mathcal{L}(f_\kappa) - \hat{\mathcal{L}}(f_\kappa)\big| > t \,\Big|\, D^{\text{train}} \Big) \\[4pt]
&\le \sum_{\kappa \in \mathcal{K}} \mathbb{P}\Big( \big|\mathcal{L}(f_\kappa) - \hat{\mathcal{L}}(f_\kappa)\big| > t \,\Big|\, D^{\text{train}} \Big)
&& \text{(union bound)} \\[4pt]
&\le \sum_{\kappa \in \mathcal{K}} 2 \exp\!\left( -\frac{2 l t^2}{b^2} \right)
&& \text{(Hoeffding)} \\[4pt]
&= 2k \exp\!\left( -\frac{2 l t^2}{b^2} \right).
\end{aligned}
$$

Hoeffding's inequality applies in the third step because, for fixed $\kappa$, the random variables $L(Z_1, f_\kappa), \dots, L(Z_l, f_\kappa)$ are iid, take values in $[0, b]$, and have mean $\mathcal{L}(f_\kappa)$.

The right-hand side does not depend on $D^{\text{train}}$, so taking expectations over $D^{\text{train}}$ (tower property) gives the unconditional bound. Finally, setting $\delta = 2k \exp(-2lt^2/b^2)$ and solving for $t$ gives $t = b\sqrt{\log(2k/\delta)/(2l)}$, which is the high-probability form. The training set size doesn't appear in the bound, because it determines how good the candidate models are and not how well the test set estimates the risk.

$\blacksquare$

### (b)
> What might go wrong if you use the same test set to select hyperparameters and to estimate the generalization error? In particular, consider the case where $k$ is large relative to $l$.

We use the notation and setup from (a), and condition on $D^{\text{train}}$ throughout, so the $f_\kappa$ are fixed.

The reported test error is optimistically biased. Using the same test set for selection and for evaluation means we report $\hat{\mathcal{L}}(\hat f) = \min_{\kappa} \hat{\mathcal{L}}(f_\kappa)$, a minimum over $k$ noisy estimates. Since each $\hat{\mathcal{L}}(f_\kappa)$ is unbiased for $\mathcal{L}(f_\kappa)$, and $\mathcal{L}(\hat f) \ge \min_\kappa \mathcal{L}(f_\kappa)$ holds pointwise,

$$
\mathbb{E}\big[\hat{\mathcal{L}}(\hat f)\big]
= \mathbb{E}\Big[\min_{\kappa} \hat{\mathcal{L}}(f_\kappa)\Big]
\le \min_{\kappa} \mathbb{E}\big[\hat{\mathcal{L}}(f_\kappa)\big]
= \min_{\kappa} \mathcal{L}(f_\kappa)
\le \mathbb{E}\big[\mathcal{L}(\hat f)\big].
$$

So on average the reported error underestimates the true risk of the selected model. 
We overfit on the test set, because chance is part of why the chosen model's test error is so low and the higher $k$ the higher the chance of a "lucky draw".

The bound from (a),

$$
\big|\mathcal{L}(\hat f) - \hat{\mathcal{L}}(\hat f)\big| \le b\sqrt{\frac{\log(2k/\delta)}{2l}},
$$

decreases like $1/\sqrt{l}$ but increases like $\sqrt{\log k}$. When $k$ is large relative to $l$, the guarantee loosens. It becomes useless ($\ge b$, which holds trivially) once $\log(2k/\delta) \ge 2l$. Because of the logarithm, this requires $k$ to be very large (exponential in $l$). Even for moderate $k$, though, we pay a $\sqrt{\log k}$ factor and, more importantly, the estimate is biased.

One solution to this problem is that we split the $l$ held-out points into a validation set of size $\alpha l$, used to select $\hat f$, and a test set of size $(1-\alpha) l$, used only to evaluate $\hat f$, with $\alpha \in (0, 1)$.

For estimation, conditional on the training and validation data, $\hat f$ is a single fixed model, and the test set is independent of it. Applying (a) with $k = 1$ gives, with probability at least $1 - \delta$,

$$
\big|\mathcal{L}(\hat f) - \hat{\mathcal{L}}_{\text{test}}(\hat f)\big| \le b\sqrt{\frac{\log(2/\delta)}{2(1-\alpha)l}},
$$

and $\hat{\mathcal{L}}_{\text{test}}(\hat f)$ is now unbiased for $\mathcal{L}(\hat f)$. The split bound is tighter than the same-set bound above exactly when

$$
\frac{\log(2/\delta)}{1-\alpha} < \log(2k/\delta)
\quad\Longleftrightarrow\quad
1 - \alpha > \frac{\log(2/\delta)}{\log(2k/\delta)}.
$$

For example, with $\delta = 0.05$ and $k = 512$, the right-hand side is $\approx 3.69 / 9.93 \approx 0.37$. The larger $k$ is, the smaller the test share needs to be for the split to pay off.

The validation set can't be made arbitrarily small, though: with fewer validation points, the selection among the $k$ candidates becomes noisier and we may pick a worse model, so the split trades off the quality of the chosen model against the accuracy of its risk estimate.

### (c)
> Commonly, we instead split the data into a training set, a test set, and a *validation* set used to select hyperparameters.
>
> Use the provided dataset on white wine (`winequality-white.csv`, a subset of the wine quality dataset) and divide it randomly into 70% train, 10% validation, and 20% test. The last column "quality" is the target, an integer from 1–10. We'll view this as a regression problem with mean squared error as the desired loss function. Pick two model classes from the set { RandomForestRegressor, KNeighborsRegressor, MLPRegressor } and read their documentation in Scikit-learn to understand the hyperparameters of these models and what they do. Do grid search over a reasonably-large set of parameters (number of parameter sets > the number of points in the validation set): fit the model on the train set for each set of parameters, and select the "best parameters" using the score on the validation set. Finally, report the performance on the test set for each of the model classes you chose. What's the relationship between the empirical risk on the validation set and the empirical risk on the test set?

**Load and split data**

In [2]:
wine = pd.read_csv("winequality-white.csv", delimiter=";")

X, y = wine.iloc[:, :-1], wine.iloc[:, -1]

X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, train_size=0.7, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_holdout, y_holdout, train_size=1/3, random_state=RANDOM_STATE)

results = {}  # filled by the grid search cells below, used for the summary table

len(y_val)

490

**Model class 1: `RandomForestRegressor`**

A random forest averages many regression trees, each fit on a bootstrap sample of the training data. Hyperparameters in the grid (all others at their defaults, e.g. 100 trees):

- `min_samples_leaf`: minimum number of training points in each leaf. Controls the bias and is a form of regularization, since a value of 1 would allow the trees to memorize each observation.
- `max_features`: number of features considered at each split. Smaller values make the trees more random and less correlated, which reduces the variance of the average. With all 11 features, this is plain bagging.
- `max_depth`: maximum depth of each tree. `None` grows the trees until the leaves are pure or `min_samples_leaf` stops the splitting.

In [3]:
param_grid = {'min_samples_leaf':[1, 2, 3, 5, 10, 15, 20, 25, 30],
              'max_features': [1, 2, 3, 5, 7, 9, 11],
              'max_depth': [2, 3, 4, 5, 6, 8, 10, 12, None]}  # all < 13 = depth of the shallowest fully grown forest, so every value gives a different model

best_val = np.inf

for parameterization in ParameterGrid(param_grid):
  model = RandomForestRegressor(**parameterization, random_state=RANDOM_STATE)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_val)
  curr_val = mean_squared_error(y_val, y_pred)

  if curr_val < best_val:
    best_val = curr_val
    best_params = parameterization
    best_model = deepcopy(model)
  
shown_params = best_params
print(f"Number of parameter sets tried: {len(ParameterGrid(param_grid))}")
print(f"Best parameters: {shown_params}")
test_mse = mean_squared_error(y_test, best_model.predict(X_test))
print(f"Validation MSE:  {best_val:.4f}")
print(f"Test MSE:        {test_mse:.4f}")

results['RandomForestRegressor'] = {'Parameter sets tried': len(ParameterGrid(param_grid)),
                                    'Best parameters': shown_params,
                                    'Validation MSE': best_val,
                                    'Test MSE': test_mse}

Number of parameter sets tried: 567
Best parameters: {'max_depth': None, 'max_features': 3, 'min_samples_leaf': 1}
Validation MSE:  0.3802
Test MSE:        0.4484


**Model class 2: `KNeighborsRegressor`**

KNN predicts the target of a point by averaging the targets of its $k$ nearest training points. Hyperparameters in the grid:

- `n_neighbors` ($k$): number of neighbors averaged. Small $k$ gives low bias but high variance; large $k$ gives smoother predictions but higher bias.
- `weights`: `'uniform'` weights all $k$ neighbors equally, `'distance'` weights them by inverse distance, so closer neighbors count more.
- `metric` (and `p` for `'minkowski'`): the distance that defines "nearest". Besides Minkowski distances with different $p$ (including $0 < p < 1$, which is not a true metric), the grid includes `chebyshev`, `braycurtis`, `canberra`, `cosine`, `correlation`, `seuclidean` (Euclidean after dividing each feature by its variance) and `mahalanobis` (additionally accounts for correlations between features). Since the features are not scaled, the choice of metric also determines how much each feature influences the distance.

In [4]:
import warnings

N_NEIGHBORS = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60]
WEIGHTS = ['uniform', 'distance']

# extra inputs some scipy metrics need (computed on the train set only)
V = X_train.var(axis=0).to_numpy()                                 # per-feature variances for 'seuclidean'
VI = np.linalg.inv(np.cov(X_train.to_numpy(), rowvar=False))       # inverse covariance for 'mahalanobis'

# list of sub-grids: 'p' only matters for minkowski, 'metric_params' only for seuclidean / mahalanobis
param_grid = [
  {'metric': ['minkowski'],
   'p': [0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 5], # for 0 < p < 1 triangle inequality doesn't hold => not a valid metric; still being tried because might work empirically
   'n_neighbors': N_NEIGHBORS, 'weights': WEIGHTS},
  {'metric': ['chebyshev', 'braycurtis', 'canberra', 'cosine', 'correlation'],
   'n_neighbors': N_NEIGHBORS, 'weights': WEIGHTS},
  {'metric': ['seuclidean'], 'metric_params': [{'V': V}],
   'n_neighbors': N_NEIGHBORS, 'weights': WEIGHTS},
  {'metric': ['mahalanobis'], 'metric_params': [{'VI': VI}],
   'n_neighbors': N_NEIGHBORS, 'weights': WEIGHTS},
]

best_val = np.inf

with warnings.catch_warnings():
  warnings.filterwarnings("ignore", message=".*Minkowski metrics are not distance metrics.*")
  for parameterization in ParameterGrid(param_grid):
    model = KNeighborsRegressor(**parameterization)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    curr_val = mean_squared_error(y_val, y_pred)

    if curr_val < best_val:
      best_val = curr_val
      best_params = parameterization
      best_model = deepcopy(model)

shown_params = {k: v for k, v in best_params.items() if k != 'metric_params'}  # hide the V / VI arrays
print(f"Number of parameter sets tried: {len(ParameterGrid(param_grid))}")
print(f"Best parameters: {shown_params}")
test_mse = mean_squared_error(y_test, best_model.predict(X_test))
print(f"Validation MSE:  {best_val:.4f}")
print(f"Test MSE:        {test_mse:.4f}")

results['KNeighborsRegressor'] = {'Parameter sets tried': len(ParameterGrid(param_grid)),
                                  'Best parameters': shown_params,
                                  'Validation MSE': best_val,
                                  'Test MSE': test_mse}

Number of parameter sets tried: 544
Best parameters: {'metric': 'seuclidean', 'n_neighbors': 15, 'weights': 'distance'}
Validation MSE:  0.4101
Test MSE:        0.4575


**Results**

In [5]:
summary = pd.DataFrame(results).T
summary['Test - Validation'] = summary['Test MSE'] - summary['Validation MSE']
summary[['Validation MSE', 'Test MSE', 'Test - Validation']] = summary[['Validation MSE', 'Test MSE', 'Test - Validation']].astype(float).round(4)
summary.drop(columns='Best parameters').rename(columns={'Parameter sets tried': 'Sets tried'})  # best parameters are printed in full above

,Sets tried,Validation MSE,Test MSE,Test - Validation
RandomForestRegressor,567,0.3802,0.4484,0.0683
KNeighborsRegressor,544,0.4101,0.4575,0.0474


**Relationship between the empirical risk on the validation set and on the test set**

The empirical risk on the validation set is lower than the empirical risk on the test set. This is consistent with the observations from (a) and (b) that by selecting the best parametrizations over many of them, the chosen parametrization's performance can partly be attributed to chance. Therefore the empirical risk on the validation set is an optimistic estimate of the true risk, for which the empirical risk on the test set is an unbiased estimator.

---
## 3.
> In class, we talked about the generalization behaviour of boosting as an increasing number of weak classifiers are included in the model fitting.

In [6]:
N_TREES = [1, 10, 100, 300, 1000]

### (a)
> Use scikit-learn to fit a gradient boosting model to dset0 (You should model the first column in `dset0.csv` given the other columns). Fit models with 1, 10, 100, 300, and 1000 trees. What do you observe about the generalization behaviour of these models?

In [7]:
dset0 = pd.read_csv("dset0-1.csv", header=None)

X, y = dset0.iloc[:, 1:], dset0.iloc[:, 0] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=RANDOM_STATE)

train_brier, test_brier = [], []

for n in N_TREES:
  model = GradientBoostingClassifier(n_estimators=n, random_state=RANDOM_STATE)
  model.fit(X_train, y_train)

  y_train_pred = model.predict_proba(X_train)[:, 1]
  y_test_pred = model.predict_proba(X_test)[:, 1]

  train_brier.append(brier_score_loss(y_train, y_train_pred))
  test_brier.append(brier_score_loss(y_test, y_test_pred))

In [8]:
performance_table = pd.DataFrame({'Number of Trees': N_TREES, 'Train Brier Score': train_brier, 'Test Brier Score': test_brier})
performance_table

,Number of Trees,Train Brier Score,Test Brier Score
0,1,2.321864e-01,0.231720
1,10,1.544010e-01,0.193936
2,100,2.097948e-02,0.116820
3,300,8.130483e-04,0.120370
4,1000,1.529040e-08,0.144026


**Observations:**

As the number of trees increases, the training Brier score keeps decreasing and is close to 0 from 300 trees onwards (model has capacity to memorize the data). The Brier score on the test set keeps decreasing till 100 trees and afterwards increases, indicating overfitting.

### (b)
> Use scikit-learn to fit a gradient boosting model to dset1 (You should model the first column in `dset1.csv` given the other columns). Fit models with 1, 10, 100, 300, and 1000 trees. What do you observe about the generalization behaviour of these models?

In [9]:
dset1 = pd.read_csv("dset1-1.csv", header=None)  # no header row in the file

X, y = dset1.iloc[:, 1:], dset1.iloc[:, 0] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=RANDOM_STATE)

train_brier, test_brier = [], []

for n in N_TREES:
  model = GradientBoostingClassifier(n_estimators=n, random_state=RANDOM_STATE)
  model.fit(X_train, y_train)

  y_train_pred = model.predict_proba(X_train)[:, 1]
  y_test_pred = model.predict_proba(X_test)[:, 1]

  train_brier.append(brier_score_loss(y_train, y_train_pred))
  test_brier.append(brier_score_loss(y_test, y_test_pred))

In [10]:
performance_table = pd.DataFrame({'Number of Trees': N_TREES, 'Train Brier Score': train_brier, 'Test Brier Score': test_brier})
performance_table

,Number of Trees,Train Brier Score,Test Brier Score
0,1,2.230215e-01,0.233987
1,10,1.236113e-01,0.147277
2,100,3.280700e-02,0.098027
3,300,2.431877e-03,0.103085
4,1000,2.627394e-07,0.126367


**Observations:**

As the number of trees increases, the training Brier score keeps decreasing and is close to 0 from 300 trees onwards (model has capacity to memorize the data). The Brier score on the test set keeps decreasing till 100 trees and afterwards increases, indicating overfitting.

### (c)
> If your observations in parts (a) and (b) differ, explain what property of the data might explain that.

In [11]:
# Correlation of the label (first column) with each feature
pd.DataFrame({'dset0': dset0.corr()[0].drop(0), 'dset1': dset1.corr()[0].drop(0)}).round(2)

,dset0,dset1
1,0.03,-0.59
2,0.20,-0.29
3,-0.11,-0.11
4,0.13,-0.03
5,-0.16,0.20


Both datasets show the same overall pattern (test Brier score lowest at around 100 trees, then increasing), but on dset1 boosting improves faster and reaches a lower test Brier score (0.098 vs. 0.117).

This can be explained by the data: in dset1, the first feature is strongly correlated with the label ($\approx -0.59$), while in dset0 no feature is strongly correlated with it (all $|\rho| \le 0.2$). So dset1 has a simpler structure that a few trees can already capture, whereas in dset0 boosting needs more trees to learn the relationship.